# Arabic Review Intelligence: Aspect-Based Sentiment Analysis with MARBERT

This notebook builds an Aspect-Based Sentiment Analysis model for Arabic and multilingual customer reviews.

Instead of predicting only one overall sentiment for a review, the model predicts sentiment for each business aspect separately, such as:

- Food
- Service
- Price
- Cleanliness
- Delivery
- Ambiance
- App experience
- General opinion

For each review, the final output is a structured JSON object containing the detected aspects and the sentiment for each detected aspect.

In [1]:
# ============================================================
# NOTEBOOK: Arabic ABSA with MARBERT
# Full end-to-end notebook code:
# - installs/imports
# - preprocessing
# - data loading
# - train/validation split
# - aspect-conditioned dataset creation
# - weighted-loss MARBERT trainer
# - evaluation
# - inference
# - JSON submission generation
#
# Expected train columns:
#   review_id
#   review_text
#   aspects
#   aspect_sentiments
#
# Expected test columns:
#   review_id
#   review_text
#
# Supported formats for train labels:
#   aspects: list or JSON string list
#   aspect_sentiments: dict or JSON string dict
#
# Output schema:
# [
#   {
#     "review_id": 23,
#     "aspects": ["service", "ambiance", "food"],
#     "aspect_sentiments": {
#       "service": "positive",
#       "ambiance": "positive",
#       "food": "negative"
#     }
#   }
# ]
# ============================================================

# ============================================================
# 1) INSTALLS
# ============================================================
# If running in a notebook, uncomment if needed:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy emoji
!pip install -q gradio

# ============================================================
# 2) IMPORTS
# ============================================================
import os
import re
import ast
import json
import math
import random
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd
import emoji

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

## 1. Project Setup and Configuration

In this section, we define the main experiment settings:

- The base model: `UBC-NLP/MARBERT`
- Output paths for the trained model and submission file
- Training hyperparameters
- The supported aspects
- The sentiment label mapping

The task is converted into a pairwise classification problem.  
For every review, we ask the model about each possible aspect and classify it as:

- `not_mentioned`
- `positive`
- `negative`
- `neutral`

In [2]:
# ============================================================
# 3) GLOBAL CONFIG
# ============================================================
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "UBC-NLP/MARBERT"
OUTPUT_DIR = "/kaggle/working/marbert_absa_output"
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.json")

MAX_LEN = 96
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 3e-5
NUM_EPOCHS = 10
WEIGHT_DECAY = 0.015
WARMUP_RATIO = 0.1
GRAD_ACCUM_STEPS = 1

USE_FP16 = torch.cuda.is_available()
USE_EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 2

DATA_DIR = "/kaggle/input/datasets/kareemmaged/deepx-nlp"

TRAIN_FILE = f"{DATA_DIR}/train_fixed(Sheet1).csv"
TEST_FILE = f"{DATA_DIR}/validation_fixed(Sheet1).csv"
# TEST_FILE = f"{DATA_DIR}/unlabeled_fixed(Sheet1).csv"
# TEST_FILE = f"{DATA_DIR}/DeepX_hidden_test (4).xlsx"

TEXT_COL = "review_text"
ID_COL = "review_id"
ASPECTS_COL = "aspects"
ASPECT_SENTIMENTS_COL = "aspect_sentiments"

ASPECTS = [
    "food",
    "service",
    "price",
    "cleanliness",
    "delivery",
    "ambiance",
    "app_experience",
    "general"
]

# If challenge requires "none" only at output time when nothing is found,
# we do NOT train "none" as a regular aspect here.
OUTPUT_NONE_LABEL = "none"

LABEL2ID = {
    "not_mentioned": 0,
    "positive": 1,
    "negative": 2,
    "neutral": 3
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

ASPECT_PROMPTS = {
    "food": "الجانب: الطعام والأكل",
    "service": "الجانب: الخدمة والتعامل",
    "price": "الجانب: السعر والأسعار",
    "cleanliness": "الجانب: النظافة",
    "delivery": "الجانب: التوصيل",
    "ambiance": "الجانب: الأجواء والمكان",
    "app_experience": "الجانب: التطبيق وتجربة الاستخدام",
    "general": "الجانب: التقييم العام"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

print("✅ Config ready")
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)

✅ Config ready
MODEL_NAME: UBC-NLP/MARBERT
OUTPUT_DIR: /kaggle/working/marbert_absa_output


## 2. Arabic Text Preprocessing

Arabic customer reviews often contain spelling variations, repeated letters, emojis, mixed punctuation, and informal dialect.

The preprocessing here is intentionally light because MARBERT was pretrained on Arabic social and dialectal text.  
We avoid aggressive cleaning such as stemming or stopword removal because those steps may remove useful sentiment signals.

The preprocessing handles:

- Arabic character normalization
- Diacritics removal
- URL, email, and mention replacement
- Repeated character reduction
- Whitespace cleanup
- Emoji preservation

In [3]:
# ============================================================
# 4) ARABIC PREPROCESSOR
# Light preprocessing suitable for MARBERT
# ============================================================
class ArabicPreprocessor:
    """
    Light Arabic preprocessing for MARBERT-based ABSA.
    Keeps punctuation, negation, and emojis by default.
    Avoids stopword removal and heavy stemming.
    """

    def __init__(
        self,
        remove_diacritics: bool = True,
        replace_urls: bool = True,
        replace_emails: bool = True,
        replace_mentions: bool = True,
        replace_hashtags: bool = False,
        keep_emojis: bool = True,
        reduce_elongation: bool = True
    ):
        self.remove_diacritics = remove_diacritics
        self.replace_urls = replace_urls
        self.replace_emails = replace_emails
        self.replace_mentions = replace_mentions
        self.replace_hashtags = replace_hashtags
        self.keep_emojis = keep_emojis
        self.reduce_elongation = reduce_elongation

        self.char_map = str.maketrans({
            'إ': 'ا',
            'أ': 'ا',
            'آ': 'ا',
            'ٱ': 'ا',
            'ى': 'ي',
            'ک': 'ك',
            'ی': 'ي',
            'ھ': 'ه',
            'ـ': '',   # tatweel
        })

        self.arabic_diacritics = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670]')
        self.control_chars = re.compile(r'[\u0000-\u001F\u007F-\u009F]')
        self.url_pattern = re.compile(r'(http\S+|www\.\S+)')
        self.email_pattern = re.compile(r'\S+@\S+')
        self.mention_pattern = re.compile(r'@\w+')
        self.hashtag_pattern = re.compile(r'#(\w+)')
        self.whitespace_pattern = re.compile(r'\s+')
        self.elongation_pattern = re.compile(r'(.)\1{2,}')
        self.repeated_punct_pattern = re.compile(r'([!?.،؛])\1{2,}')

    def normalize_unicode(self, text: str) -> str:
        return unicodedata.normalize("NFKC", text)

    def normalize_arabic(self, text: str) -> str:
        text = text.translate(self.char_map)
        if self.remove_diacritics:
            text = self.arabic_diacritics.sub('', text)
        return text

    def reduce_repeated_chars(self, text: str) -> str:
        if self.reduce_elongation:
            text = self.elongation_pattern.sub(r'\1\1', text)
        text = self.repeated_punct_pattern.sub(r'\1\1', text)
        return text

    def replace_special_tokens(self, text: str) -> str:
        if self.replace_urls:
            text = self.url_pattern.sub(' <URL> ', text)
        if self.replace_emails:
            text = self.email_pattern.sub(' <EMAIL> ', text)
        if self.replace_mentions:
            text = self.mention_pattern.sub(' <USER> ', text)
        if self.replace_hashtags:
            text = self.hashtag_pattern.sub(r' \1 ', text)
        return text

    def handle_emojis(self, text: str) -> str:
        if self.keep_emojis:
            return text
        return emoji.replace_emoji(text, replace=' ')

    def clean(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        text = self.normalize_unicode(text)
        text = self.control_chars.sub(' ', text)
        text = self.replace_special_tokens(text)
        text = self.normalize_arabic(text)
        text = self.reduce_repeated_chars(text)
        text = self.handle_emojis(text)
        text = self.whitespace_pattern.sub(' ', text).strip()
        return text

    def full_preprocess(self, text: str) -> str:
        return self.clean(text)


preprocessor = ArabicPreprocessor()

print("✅ Preprocessor ready")
print(preprocessor.full_preprocess("الأكــل رائع 😍 لكن الأسعــار غالية جداً، والخدمة ليست جيدة!!!"))

✅ Preprocessor ready
الاكل رائع 😍 لكن الاسعار غالية جدا، والخدمة ليست جيدة!!


## 3. Parsing Aspect and Sentiment Labels

The labeled dataset stores aspects and aspect sentiments as list/dictionary-like values.

This section safely converts those values into Python objects:

- `aspects` becomes a list of mentioned aspects.
- `aspect_sentiments` becomes a dictionary mapping each aspect to its sentiment.

Invalid or unsupported aspects are ignored to keep the training data clean.

In [4]:
# ============================================================
# 5) HELPER FUNCTIONS FOR SAFE PARSING
# ============================================================
def safe_parse_list(x):
    """
    Converts value to a Python list safely.
    Supports:
    - already-list
    - JSON string list
    - Python-literal string list
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            return json.loads(x)
        except Exception:
            try:
                return ast.literal_eval(x)
            except Exception:
                return [x]
    return []


def safe_parse_dict(x):
    """
    Converts value to a Python dict safely.
    Supports:
    - already-dict
    - JSON string dict
    - Python-literal string dict
    """
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except Exception:
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
    return {}


def normalize_gold_aspects(aspects: List[str]) -> List[str]:
    cleaned = []
    for a in aspects:
        if not isinstance(a, str):
            continue
        a = a.strip()
        if a in ASPECTS:
            cleaned.append(a)
    return cleaned


def normalize_gold_sentiments(d: Dict[str, str]) -> Dict[str, str]:
    out = {}
    for k, v in d.items():
        if k not in ASPECTS:
            continue
        if v not in LABEL2ID:
            # only keep positive/negative/neutral/not_mentioned if present
            # if malformed, skip
            continue
        out[k] = v
    return out

## 4. Loading the Data

The project uses three types of data:

1. Labeled training data
2. Labeled validation data
3. Unlabeled or hidden test data

The labeled files contain:

- `review_id`
- `review_text`
- metadata such as rating, date, business, category, and platform
- `aspects`
- `aspect_sentiments`

The hidden or unlabeled files contain only the review and metadata columns, so the model must predict the aspects and sentiments.

In [5]:
# ============================================================
# 6) LOAD DATA
# ============================================================
def read_table(path: str):
    if path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".xlsx") or path.endswith(".xls"):
        return pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")


def load_data(train_path: str, test_path: Optional[str] = None):
    train_df = read_table(train_path)
    test_df = read_table(test_path) if test_path and os.path.exists(test_path) else None
    return train_df, test_df


train_df, test_df = load_data(TRAIN_FILE, TEST_FILE)

print("✅ Data loaded")
print("Train shape:", train_df.shape)
if test_df is not None:
    print("Test shape:", test_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
if test_df is not None:
    print("Test columns:", test_df.columns.tolist())


# ============================================================
# 7) VALIDATE / CLEAN DATAFRAME SCHEMA
# ============================================================
required_train_cols = [ID_COL, TEXT_COL, ASPECTS_COL, ASPECT_SENTIMENTS_COL]
for c in required_train_cols:
    if c not in train_df.columns:
        raise ValueError(f"Missing required train column: {c}")

required_test_cols = [ID_COL, TEXT_COL]
if test_df is not None:
    for c in required_test_cols:
        if c not in test_df.columns:
            raise ValueError(f"Missing required test column: {c}")

train_df = train_df.copy()
train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)
train_df[ID_COL] = train_df[ID_COL].astype(int)

if test_df is not None:
    test_df = test_df.copy()
    test_df[TEXT_COL] = test_df[TEXT_COL].fillna("").astype(str)
    test_df[ID_COL] = test_df[ID_COL].astype(int)

print("✅ Schema validated")


# ============================================================
# 8) PARSE LABEL COLUMNS
# ============================================================
train_df["parsed_aspects"] = train_df[ASPECTS_COL].apply(safe_parse_list).apply(normalize_gold_aspects)
train_df["parsed_aspect_sentiments"] = train_df[ASPECT_SENTIMENTS_COL].apply(safe_parse_dict).apply(normalize_gold_sentiments)

print("✅ Parsed labels")
print(train_df[[ID_COL, TEXT_COL, "parsed_aspects", "parsed_aspect_sentiments"]].head(3))


# ============================================================
# 9) PREPROCESS TEXT
# ============================================================
train_df["clean_text"] = train_df[TEXT_COL].apply(preprocessor.full_preprocess)
if test_df is not None:
    test_df["clean_text"] = test_df[TEXT_COL].apply(preprocessor.full_preprocess)

print("✅ Text preprocessed")
print(train_df[[TEXT_COL, "clean_text"]].head(3))

✅ Data loaded
Train shape: (1971, 9)
Test shape: (500, 9)

Train columns: ['review_id', 'review_text', 'star_rating', 'date', 'business_name', 'business_category', 'platform', 'aspects', 'aspect_sentiments']
Test columns: ['review_id', 'review_text', 'star_rating', 'date', 'business_name', 'business_category', 'platform', 'aspects', 'aspect_sentiments']
✅ Schema validated
✅ Parsed labels
   review_id                                        review_text  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
1       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...   
2       1975  تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...   

                     parsed_aspects  \
0        [app_experience, delivery]   
1  [cleanliness, ambiance, service]   
2         [service, delivery, food]   

                            parsed_aspect_sentiments  
0  {'app_experience': 'negative', 'delivery': 'ne...  
1  {'cleanliness': 'positive', 'ambiance': 'posit...  
2  {'service': 'ne

## 5. Converting Reviews into Aspect-Level Training Rows

This is the key modeling idea.

Each original review is expanded into multiple rows: one row for each possible aspect.

For example, if the supported aspects are:

`food`, `service`, `price`, `cleanliness`, `delivery`, `ambiance`, `app_experience`, and `general`

then one review becomes eight training examples.

If an aspect appears in the review, the label is its sentiment.  
If it does not appear, the label is `not_mentioned`.

This allows the model to learn both:

- whether an aspect is mentioned
- what sentiment is expressed toward that aspect

In [6]:
# ============================================================
# 10) BUILD PAIRWISE TRAINING DATA
# One row per (review, aspect)
# label in {not_mentioned, positive, negative, neutral}
# ============================================================
def make_pairwise_training_df(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        review_id = int(row[ID_COL])
        review = row["clean_text"]
        gold_aspects = row["parsed_aspects"]
        gold_sentiments = row["parsed_aspect_sentiments"]

        for aspect in ASPECTS:
            if aspect in gold_aspects:
                label_name = gold_sentiments.get(aspect, "neutral")
                if label_name not in LABEL2ID:
                    label_name = "neutral"
            else:
                label_name = "not_mentioned"

            rows.append({
                "review_id": review_id,
                "review_text": review,
                "aspect": aspect,
                "aspect_prompt": ASPECT_PROMPTS[aspect],
                "label_name": label_name,
                "label": LABEL2ID[label_name]
            })

    return pd.DataFrame(rows)


pair_df = make_pairwise_training_df(train_df)
print("✅ Pairwise training data created")
print("Pairwise shape:", pair_df.shape)
print(pair_df.head(10))



✅ Pairwise training data created
Pairwise shape: (15768, 6)
   review_id                                        review_text  \
0       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
1       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
2       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
3       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
4       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
5       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
6       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
7       7238                لا يوجد الدفع بالبطاقه عند الاستلام   
8       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...   
9       1036  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...   

           aspect                     aspect_prompt     label_name  label  
0            food             الجانب: الطعام والأكل  not_mentioned      0  
1         service           الجانب: الخدمة والتعام

## 6. Train/Validation Split

The split is done by original `review_id`, not by aspect-level rows.

This is important because if rows from the same review appeared in both training and validation, the validation score would be inflated due to leakage.

After the split, each review belongs entirely to either the training set or the validation set.

In [7]:
# ============================================================
# 11) TRAIN / VALID SPLIT
# Split by original reviews to prevent leakage
# ============================================================
train_ids, valid_ids = train_test_split(
    train_df[ID_COL].tolist(),
    test_size=0.15,
    random_state=SEED,
    shuffle=True
)

train_pair_df = pair_df[pair_df["review_id"].isin(train_ids)].reset_index(drop=True)
valid_pair_df = pair_df[pair_df["review_id"].isin(valid_ids)].reset_index(drop=True)

print("✅ Train/valid split done")
print("Train pair shape:", train_pair_df.shape)
print("Valid pair shape:", valid_pair_df.shape)


# ============================================================
# 12) LABEL DISTRIBUTION
# ============================================================
print("Train label distribution:")
print(train_pair_df["label_name"].value_counts())

print("\nValid label distribution:")
print(valid_pair_df["label_name"].value_counts())

✅ Train/valid split done
Train pair shape: (13400, 6)
Valid pair shape: (2368, 6)
Train label distribution:
label_name
not_mentioned    10627
positive          1388
negative          1316
neutral             69
Name: count, dtype: int64

Valid label distribution:
label_name
not_mentioned    1865
positive          258
negative          222
neutral            23
Name: count, dtype: int64


## 7. Handling Class Imbalance

Most aspect-review pairs are labeled as `not_mentioned`, because each review usually talks about only a few aspects.

To reduce the effect of this imbalance, we use class weights in the loss function.

This gives more importance to rare classes such as `neutral`, while preventing the model from simply predicting `not_mentioned` too often.

In [8]:
# ============================================================
# 13) CLASS WEIGHTS
# Weighted loss to handle imbalance
# ============================================================
def compute_class_weights(labels: List[int], num_classes: int):
    counts = np.bincount(labels, minlength=num_classes)
    total = counts.sum()
    weights = total / (num_classes * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_class_weights(train_pair_df["label"].tolist(), num_classes=len(LABEL2ID))
print("✅ Class weights:", class_weights)


# ============================================================
# 14) TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer loaded")



✅ Class weights: tensor([0.0234, 0.1794, 0.1892, 3.6080])


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ Tokenizer loaded


## 8. Model Training with MARBERT

We fine-tune `UBC-NLP/MARBERT`, a transformer model designed for Arabic and dialectal Arabic text.

Each model input contains two parts:

1. The customer review text
2. An aspect-specific Arabic prompt

For example:

Review:  
`الخدمة ممتازة والأكل سيء`

Aspect prompt:  
`الجانب: الطعام والأكل`

The model then predicts the sentiment label for that review-aspect pair.

In [12]:
# ============================================================
# 15) DATASET
# ============================================================
class ABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = 160):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["review_text"]
        aspect_prompt = row["aspect_prompt"]
        label = int(row["label"])

        enc = self.tokenizer(
            text,
            aspect_prompt,
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )

        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


train_dataset = ABSADataset(train_pair_df, tokenizer, max_len=MAX_LEN)
valid_dataset = ABSADataset(valid_pair_df, tokenizer, max_len=MAX_LEN)

print("✅ Datasets ready")
print("Train dataset len:", len(train_dataset))
print("Valid dataset len:", len(valid_dataset))


# ============================================================
# 16) MODEL
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.3,
)

print("✅ Model loaded")
print(model.config)


# ============================================================
# 17) CUSTOM TRAINER WITH WEIGHTED LOSS
# ============================================================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        return (loss, outputs) if return_outputs else loss

✅ Datasets ready
Train dataset len: 13400
Valid dataset len: 2368


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

✅ Model loaded
BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.3,
  "bos_token_id": null,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "not_mentioned",
    "1": "positive",
    "2": "negative",
    "3": "neutral"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "negative": 2,
    "neutral": 3,
    "not_mentioned": 0,
    "positive": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",


## 9. Validation Evaluation

The model is evaluated on the aspect-level validation set using:

- Accuracy
- Macro F1
- Weighted F1
- Classification report
- Confusion matrix

Macro F1 is especially important because the dataset is imbalanced and we want the model to perform well across all sentiment classes, not only the majority class.

In [13]:
# ============================================================
# 18) METRICS
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }


# ============================================================
# 19) TRAINING ARGUMENTS
# ============================================================
opt_model = {'learning_rate': 1.9059880595407877e-05, 'weight_decay': 0.07241477030927881, 'warmup_ratio': 0.0921903431405731, 
'hidden_dropout_prob': 0.1, 'attention_probs_dropout_prob': 0.3, 'batch_size': 16, 'num_train_epochs': 6}
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # overwrite_output_dir=True,
    learning_rate=1.9059880595407877e-05,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=6,
    weight_decay=0.07241477030927881,
    warmup_ratio=0.0921903431405731,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=USE_FP16,
    report_to="none",
    # hidden_dropout_prob=0.3,  # default is 0.1
    # attention_probs_dropout_prob=0.3,
    seed=SEED
)

callbacks = []
if USE_EARLY_STOPPING:
    callbacks.append(EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE))

print("✅ Training arguments ready")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Training arguments ready


In [14]:
# ============================================================
# 20) TRAINER
# ============================================================
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=callbacks
)

print("✅ Trainer ready")


# ============================================================
# 21) TRAIN
# ============================================================
train_result = trainer.train()

print("✅ Training complete")
print(train_result)

✅ Trainer ready


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.955003,0.927552,0.614865,0.421722,0.652461
2,0.637618,0.934467,0.869510,0.584072,0.873062
3,0.589625,0.975220,0.878378,0.648238,0.882349
4,0.374870,0.885881,0.901182,0.732145,0.904527
5,0.283084,1.391442,0.915118,0.716467,0.914562
6,0.159558,1.257566,0.923142,0.733501,0.922464


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✅ Training complete
TrainOutput(global_step=2514, training_loss=0.5319672631382847, metrics={'train_runtime': 1200.3108, 'train_samples_per_second': 66.983, 'train_steps_per_second': 2.094, 'total_flos': 3966470384947200.0, 'train_loss': 0.5319672631382847, 'epoch': 6.0})


In [15]:
# ============================================================
# 22) SAVE BEST MODEL
# ============================================================
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("✅ Best model saved to:", 'marbert_absa_output/opt1')


# ============================================================
# 23) EVALUATE ON VALID
# ============================================================
eval_results = trainer.evaluate(eval_dataset=valid_dataset)
print("✅ Validation results")
print(eval_results)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved to: marbert_absa_output/opt1


✅ Validation results
{'eval_loss': 1.2575658559799194, 'eval_accuracy': 0.9231418918918919, 'eval_macro_f1': 0.7335012233791041, 'eval_weighted_f1': 0.9224637901018534, 'eval_runtime': 10.3025, 'eval_samples_per_second': 229.848, 'eval_steps_per_second': 3.591, 'epoch': 6.0}


In [16]:
# ============================================================
# 24) DETAILED VALIDATION REPORT
# ============================================================
pred_output = trainer.predict(valid_dataset)
valid_logits = pred_output.predictions
valid_labels = pred_output.label_ids
valid_preds = np.argmax(valid_logits, axis=1)

print("Classification report on pairwise validation set:")
print(
    classification_report(
        valid_labels,
        valid_preds,
        target_names=[ID2LABEL[i] for i in range(len(ID2LABEL))],
        digits=4
    )
)

print("Confusion matrix:")
print(confusion_matrix(valid_labels, valid_preds))


# ============================================================
# 25) INFERENCE HELPERS
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer.model.to(device)
trainer.model.eval()

def softmax_np(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)

@torch.no_grad()
def predict_aspect_sentiment_for_review(
    review_text: str,
    tokenizer,
    model,
    max_len: int = 160
) -> Dict[str, Dict[str, Any]]:
    """
    Returns per-aspect predictions with probabilities.
    """
    clean_review = preprocessor.full_preprocess(review_text)
    results = {}

    for aspect in ASPECTS:
        prompt = ASPECT_PROMPTS[aspect]

        enc = tokenizer(
            clean_review,
            prompt,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        logits = outputs.logits.detach().cpu().numpy()[0]
        probs = softmax_np(logits)

        pred_id = int(np.argmax(probs))
        pred_label = ID2LABEL[pred_id]

        results[aspect] = {
            "pred_id": pred_id,
            "pred_label": pred_label,
            "probs": {
                ID2LABEL[i]: float(probs[i]) for i in range(len(probs))
            }
        }

    return results

Classification report on pairwise validation set:
               precision    recall  f1-score   support

not_mentioned     0.9606    0.9539    0.9572      1865
     positive     0.8377    0.8605    0.8489       258
     negative     0.7458    0.8063    0.7749       222
      neutral     0.5455    0.2609    0.3529        23

     accuracy                         0.9231      2368
    macro avg     0.7724    0.7204    0.7335      2368
 weighted avg     0.9230    0.9231    0.9225      2368

Confusion matrix:
[[1779   31   54    1]
 [  28  222    5    3]
 [  38    4  179    1]
 [   7    8    2    6]]


## 10. Review-Level Prediction and Threshold Tuning

After predicting sentiment for every aspect, we convert aspect-level predictions back into a review-level JSON output.

The model estimates whether each aspect is mentioned using:

`mention_confidence = 1 - P(not_mentioned)`

If the confidence is lower than the selected threshold, the aspect is removed from the final prediction.

Threshold tuning helps control how strict the model should be when deciding whether an aspect is actually mentioned.

In [17]:
# ============================================================
# 26) OPTIONAL THRESHOLD TUNING ON VALID SET
# We tune a mention threshold:
# mention_conf = 1 - P(not_mentioned)
# If mention_conf < threshold => aspect = not_mentioned
# ============================================================
def build_validation_review_level_df(valid_original_df: pd.DataFrame) -> pd.DataFrame:
    cols = [ID_COL, "clean_text", "parsed_aspects", "parsed_aspect_sentiments"]
    return valid_original_df[valid_original_df[ID_COL].isin(valid_ids)][cols].reset_index(drop=True)

valid_review_df = build_validation_review_level_df(train_df)

@torch.no_grad()
def predict_review_with_threshold(
    review_text: str,
    tokenizer,
    model,
    threshold: float = 0.5,
    max_len: int = 160
):
    per_aspect = predict_aspect_sentiment_for_review(
        review_text=review_text,
        tokenizer=tokenizer,
        model=model,
        max_len=max_len
    )

    found_aspects = []
    aspect_sentiments = {}

    for aspect, info in per_aspect.items():
        p_not = info["probs"]["not_mentioned"]
        mention_conf = 1.0 - p_not

        pred_label = info["pred_label"]

        if pred_label != "not_mentioned" and mention_conf >= threshold:
            found_aspects.append(aspect)
            aspect_sentiments[aspect] = pred_label

    if len(found_aspects) == 0:
        found_aspects = [OUTPUT_NONE_LABEL]
        aspect_sentiments = {}

    return {
        "aspects": found_aspects,
        "aspect_sentiments": aspect_sentiments
    }


def score_review_level_macro_f1(valid_reviews_df, tokenizer, model, threshold=0.5, max_len=160):
    """
    Approximate review-level aspect-sentiment score by flattening (review, aspect) labels.
    """
    y_true = []
    y_pred = []

    for _, row in valid_reviews_df.iterrows():
        gold_aspects = row["parsed_aspects"]
        gold_sentiments = row["parsed_aspect_sentiments"]
        pred = predict_review_with_threshold(
            review_text=row["clean_text"],
            tokenizer=tokenizer,
            model=model,
            threshold=threshold,
            max_len=max_len
        )

        pred_aspects = pred["aspects"]
        pred_sentiments = pred["aspect_sentiments"]

        for aspect in ASPECTS:
            if aspect in gold_aspects:
                true_label = gold_sentiments.get(aspect, "neutral")
            else:
                true_label = "not_mentioned"

            if aspect in pred_aspects:
                pred_label = pred_sentiments.get(aspect, "neutral")
            else:
                pred_label = "not_mentioned"

            y_true.append(LABEL2ID[true_label])
            y_pred.append(LABEL2ID[pred_label])

    score = f1_score(y_true, y_pred, average="macro")
    return score


threshold_candidates = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
best_threshold = 0.5
best_score = -1

print("🔍 Tuning threshold on validation set...")
for th in threshold_candidates:
    score = score_review_level_macro_f1(
        valid_reviews_df=valid_review_df,
        tokenizer=tokenizer,
        model=trainer.model,
        threshold=th,
        max_len=MAX_LEN
    )
    print(f"Threshold={th:.2f} -> review-level macro F1={score:.5f}")
    if score > best_score:
        best_score = score
        best_threshold = th

print("✅ Best threshold:", best_threshold)
print("✅ Best review-level macro F1:", best_score)


# ============================================================
# 27) REVIEW-LEVEL PREDICTION FOR A SINGLE EXAMPLE
# ============================================================
sample_text = "الخدمة ممتازة والأجواء جميلة لكن الأكل سيء"
sample_pred = predict_review_with_threshold(
    review_text=sample_text,
    tokenizer=tokenizer,
    model=trainer.model,
    threshold=best_threshold,
    max_len=MAX_LEN
)
print("✅ Sample prediction")
print(json.dumps(sample_pred, ensure_ascii=False, indent=2))


# ============================================================
# 28) PREDICT WHOLE TEST SET
# ============================================================
def predict_dataframe(
    df: pd.DataFrame,
    tokenizer,
    model,
    threshold: float = 0.5,
    max_len: int = 160
) -> List[Dict[str, Any]]:
    predictions = []

    for _, row in df.iterrows():
        review_id = int(row[ID_COL])
        review_text = row[TEXT_COL]

        pred = predict_review_with_threshold(
            review_text=review_text,
            tokenizer=tokenizer,
            model=model,
            threshold=threshold,
            max_len=max_len
        )

        predictions.append({
            "review_id": review_id,
            "aspects": pred["aspects"],
            "aspect_sentiments": pred["aspect_sentiments"]
        })

    return predictions


if test_df is not None:
    test_predictions = predict_dataframe(
        df=test_df,
        tokenizer=tokenizer,
        model=trainer.model,
        threshold=best_threshold,
        max_len=MAX_LEN
    )

    print("✅ Test predictions generated")
    print(json.dumps(test_predictions[:3], ensure_ascii=False, indent=2))

🔍 Tuning threshold on validation set...
Threshold=0.30 -> review-level macro F1=0.73350
Threshold=0.35 -> review-level macro F1=0.73350
Threshold=0.40 -> review-level macro F1=0.73350
Threshold=0.45 -> review-level macro F1=0.73350
Threshold=0.50 -> review-level macro F1=0.73350
Threshold=0.55 -> review-level macro F1=0.73374
Threshold=0.60 -> review-level macro F1=0.73423
Threshold=0.65 -> review-level macro F1=0.73611
✅ Best threshold: 0.65
✅ Best review-level macro F1: 0.736108850346013
✅ Sample prediction
{
  "aspects": [
    "food",
    "service",
    "ambiance"
  ],
  "aspect_sentiments": {
    "food": "negative",
    "service": "positive",
    "ambiance": "positive"
  }
}
✅ Test predictions generated
[
  {
    "review_id": 4446,
    "aspects": [
      "none"
    ],
    "aspect_sentiments": {}
  },
  {
    "review_id": 8612,
    "aspects": [
      "price",
      "app_experience"
    ],
    "aspect_sentiments": {
      "price": "positive",
      "app_experience": "positive"
    }


## 11. Submission File Generation

The final output format is a JSON list.

Each item contains:

- `review_id`
- detected `aspects`
- `aspect_sentiments`

If no aspect is detected, the output uses:

```json
{
  "aspects": ["none"],
  "aspect_sentiments": {}
}
```

In [18]:
# ============================================================
# 29) SAVE SUBMISSION JSON
# ============================================================
def save_predictions_json(predictions: List[Dict[str, Any]], output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

if test_df is not None:
    save_predictions_json(test_predictions, SUBMISSION_PATH)
    print("✅ Submission saved to:", SUBMISSION_PATH)

✅ Submission saved to: /kaggle/working/marbert_absa_output/submission.json


In [19]:
# ============================================================
# 30) OPTIONAL: PREVIEW SUBMISSION FILE
# ============================================================
if test_df is not None and os.path.exists(SUBMISSION_PATH):
    with open(SUBMISSION_PATH, "r", encoding="utf-8") as f:
        loaded_preview = json.load(f)

    print("✅ Submission preview:")
    print(json.dumps(loaded_preview[:5], ensure_ascii=False, indent=2))


# ============================================================
# 31) OPTIONAL: VALIDATION REVIEW-LEVEL JSON PREVIEW
# This helps inspect the model behavior before test submission
# ============================================================
def predict_validation_preview(valid_reviews_df, tokenizer, model, threshold=0.5, n=10):
    rows = []
    for _, row in valid_reviews_df.head(n).iterrows():
        pred = predict_review_with_threshold(
            review_text=row["clean_text"],
            tokenizer=tokenizer,
            model=model,
            threshold=threshold,
            max_len=MAX_LEN
        )

        rows.append({
            "review_id": int(row[ID_COL]),
            "gold_aspects": row["parsed_aspects"],
            "gold_aspect_sentiments": row["parsed_aspect_sentiments"],
            "pred_aspects": pred["aspects"],
            "pred_aspect_sentiments": pred["aspect_sentiments"],
            "review_text": row["clean_text"]
        })
    return rows

valid_preview = predict_validation_preview(
    valid_reviews_df=valid_review_df,
    tokenizer=tokenizer,
    model=trainer.model,
    threshold=best_threshold,
    n=10
)

print("✅ Validation preview:")
print(json.dumps(valid_preview[:3], ensure_ascii=False, indent=2))


# ============================================================
# 32) OPTIONAL: LOAD SAVED MODEL LATER
# ============================================================
def load_saved_model(model_dir: str):
    tok = AutoTokenizer.from_pretrained(model_dir)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_dir)
    mdl.to(device)
    mdl.eval()
    return tok, mdl

# Example:
# reloaded_tokenizer, reloaded_model = load_saved_model(BEST_MODEL_DIR)

✅ Submission preview:
[
  {
    "review_id": 4446,
    "aspects": [
      "none"
    ],
    "aspect_sentiments": {}
  },
  {
    "review_id": 8612,
    "aspects": [
      "price",
      "app_experience"
    ],
    "aspect_sentiments": {
      "price": "positive",
      "app_experience": "positive"
    }
  },
  {
    "review_id": 6729,
    "aspects": [
      "service",
      "price"
    ],
    "aspect_sentiments": {
      "service": "negative",
      "price": "negative"
    }
  },
  {
    "review_id": 6292,
    "aspects": [
      "general"
    ],
    "aspect_sentiments": {
      "general": "negative"
    }
  },
  {
    "review_id": 1639,
    "aspects": [
      "service",
      "ambiance"
    ],
    "aspect_sentiments": {
      "service": "positive",
      "ambiance": "positive"
    }
  }
]
✅ Validation preview:
[
  {
    "review_id": 832,
    "gold_aspects": [
      "food",
      "service"
    ],
    "gold_aspect_sentiments": {
      "food": "negative",
      "service": "negative"
    }

## 12. Custom Testing and Demo Preparation

This section tests the trained model on custom examples.

The goal is to inspect how the model behaves on realistic customer reviews, including:

- Arabic dialect
- Mixed Arabic and English
- Long reviews with multiple opinions
- Positive and negative sentiment in the same review
- Informal customer language

These examples are also useful for preparing a public demo.

In [20]:
# ============================================================
# 33) OPTIONAL: PREDICT CUSTOM REVIEWS
# ============================================================
custom_reviews = [
    {"review_id": 23, "review_text": "الخدمة ممتازة والأجواء جميلة لكن الأكل سيء"},
    {"review_id": 25, "review_text": "التوصيل تأخر جدا وتعامل الموظفين كان سيئا"},
    {"review_id": 65, "review_text": "المكان رائع والأجواء مريحة جدا"}
]

custom_df = pd.DataFrame(custom_reviews)

custom_predictions = predict_dataframe(
    df=custom_df,
    tokenizer=tokenizer,
    model=trainer.model,
    threshold=best_threshold,
    max_len=MAX_LEN
)

print("✅ Custom predictions:")
print(json.dumps(custom_predictions, ensure_ascii=False, indent=2))


# ============================================================
# 34) OPTIONAL: ENFORCE ASPECT ORDER
# If you want a stable order like your sample output
# ============================================================
ASPECT_ORDER = {
    "service": 0,
    "ambiance": 1,
    "food": 2,
    "delivery": 3,
    "price": 4,
    "cleanliness": 5,
    "app_experience": 6,
    "general": 7,
    "none": 999
}

def sort_prediction_aspects(predictions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    sorted_preds = []
    for item in predictions:
        aspects = item["aspects"]
        if aspects == [OUTPUT_NONE_LABEL]:
            sorted_aspects = aspects
            sorted_sentiments = item["aspect_sentiments"]
        else:
            sorted_aspects = sorted(aspects, key=lambda x: ASPECT_ORDER.get(x, 999))
            sorted_sentiments = {a: item["aspect_sentiments"][a] for a in sorted_aspects}

        sorted_preds.append({
            "review_id": item["review_id"],
            "aspects": sorted_aspects,
            "aspect_sentiments": sorted_sentiments
        })
    return sorted_preds

custom_predictions_sorted = sort_prediction_aspects(custom_predictions)
print("✅ Custom predictions sorted:")
print(json.dumps(custom_predictions_sorted, ensure_ascii=False, indent=2))

if test_df is not None:
    test_predictions_sorted = sort_prediction_aspects(test_predictions)
    save_predictions_json(test_predictions_sorted, SUBMISSION_PATH)
    print("✅ Sorted submission overwritten to:", SUBMISSION_PATH)


# ============================================================
# 35) OPTIONAL: SAVE TRAINING ARTIFACTS
# ============================================================
artifacts = {
    "model_name": MODEL_NAME,
    "max_len": MAX_LEN,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "best_threshold": best_threshold,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "aspects": ASPECTS,
    "aspect_prompts": ASPECT_PROMPTS
}

with open(os.path.join(OUTPUT_DIR, "artifacts.json"), "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("✅ Artifacts saved")

✅ Custom predictions:
[
  {
    "review_id": 23,
    "aspects": [
      "food",
      "service",
      "ambiance"
    ],
    "aspect_sentiments": {
      "food": "negative",
      "service": "positive",
      "ambiance": "positive"
    }
  },
  {
    "review_id": 25,
    "aspects": [
      "service",
      "delivery"
    ],
    "aspect_sentiments": {
      "service": "negative",
      "delivery": "negative"
    }
  },
  {
    "review_id": 65,
    "aspects": [
      "ambiance"
    ],
    "aspect_sentiments": {
      "ambiance": "positive"
    }
  }
]
✅ Custom predictions sorted:
[
  {
    "review_id": 23,
    "aspects": [
      "service",
      "ambiance",
      "food"
    ],
    "aspect_sentiments": {
      "service": "positive",
      "ambiance": "positive",
      "food": "negative"
    }
  },
  {
    "review_id": 25,
    "aspects": [
      "service",
      "delivery"
    ],
    "aspect_sentiments": {
      "service": "negative",
      "delivery": "negative"
    }
  },
  {
    "review

## 13. Final Check

This section confirms that the full pipeline completed successfully.

It reports:

- The best saved model directory
- The submission file path
- The selected threshold
- The validation macro F1 score

The saved model can now be used for inference or connected to a Gradio demo.

In [21]:
# ============================================================
# 36) FINAL CHECK
# ============================================================
print("\n🎯 Pipeline complete")
print("Best model dir:", BEST_MODEL_DIR)
if test_df is not None:
    print("Submission path:", SUBMISSION_PATH)
print("Best threshold:", best_threshold)
print("Validation macro F1:", eval_results.get("eval_macro_f1", None))


🎯 Pipeline complete
Best model dir: /kaggle/working/marbert_absa_output/best_model
Submission path: /kaggle/working/marbert_absa_output/submission.json
Best threshold: 0.65
Validation macro F1: 0.7335012233791041


## 14. Interactive Gradio Demo

This section launches a simple web interface for testing the trained ABSA model.

The demo is designed for non-technical audiences.  
A user can paste a customer review, click Analyze, and see:

- Detected aspects
- Sentiment for each aspect
- Confidence scores
- Final JSON output

In [ ]:
"""
Gradio demo for Arabic / multilingual Aspect-Based Sentiment Analysis (ABSA).

How to use:
1) Train the notebook first so this folder exists: ./marbert_absa_output/best_model
2) Make sure artifacts.json exists in ./marbert_absa_output/ (your notebook saves it in section 35).
3) Install Gradio if needed: pip install -q gradio
4) Run: python gradio_absa_demo_app.py
"""

import os
import re
import ast
import json
import html
import unicodedata
from typing import Dict, Any, List

import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

try:
    import emoji
except Exception:  # keeps the app usable if emoji is not installed
    emoji = None


# -----------------------------
# Paths
# -----------------------------
OUTPUT_DIR = "/kaggle/working/marbert_absa_output"
MODEL_DIR = os.path.join(OUTPUT_DIR, "best_model")
ARTIFACTS_PATH = os.path.join(OUTPUT_DIR, "artifacts.json")

if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(
        f"Could not find {MODEL_DIR}. Train the notebook first, then run this app."
    )


# -----------------------------
# Load saved training artifacts
# -----------------------------
def load_artifacts() -> Dict[str, Any]:
    defaults = {
        "max_len": 96,
        "best_threshold": 0.30,
        "label2id": {
            "not_mentioned": 0,
            "positive": 1,
            "negative": 2,
            "neutral": 3,
        },
        "id2label": {
            0: "not_mentioned",
            1: "positive",
            2: "negative",
            3: "neutral",
        },
        "aspects": [
            "food",
            "service",
            "price",
            "cleanliness",
            "delivery",
            "ambiance",
            "app_experience",
            "general",
        ],
        "aspect_prompts": {
            "food": "الجانب: الطعام والأكل",
            "service": "الجانب: الخدمة والتعامل",
            "price": "الجانب: السعر والأسعار",
            "cleanliness": "الجانب: النظافة",
            "delivery": "الجانب: التوصيل",
            "ambiance": "الجانب: الأجواء والمكان",
            "app_experience": "الجانب: التطبيق وتجربة الاستخدام",
            "general": "الجانب: التقييم العام",
        },
    }

    if not os.path.exists(ARTIFACTS_PATH):
        return defaults

    with open(ARTIFACTS_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)

    defaults.update(saved)

    # JSON saves dictionary keys as strings; convert id2label keys back to int.
    defaults["id2label"] = {int(k): v for k, v in defaults["id2label"].items()}
    return defaults


artifacts = load_artifacts()
MAX_LEN = int(artifacts["max_len"])
DEFAULT_THRESHOLD = float(artifacts.get("best_threshold", 0.30))
ASPECTS = artifacts["aspects"]
ASPECT_PROMPTS = artifacts["aspect_prompts"]
LABEL2ID = artifacts["label2id"]
ID2LABEL = artifacts["id2label"]
OUTPUT_NONE_LABEL = "none"

ASPECT_DISPLAY = {
    "food": "Food / الطعام",
    "service": "Service / الخدمة",
    "price": "Price / السعر",
    "cleanliness": "Cleanliness / النظافة",
    "delivery": "Delivery / التوصيل",
    "ambiance": "Ambiance / الأجواء",
    "app_experience": "App Experience / تجربة التطبيق",
    "general": "General / عام",
}

SENTIMENT_DISPLAY = {
    "positive": "Positive 👍",
    "negative": "Negative 👎",
    "neutral": "Neutral 😐",
    "not_mentioned": "Not mentioned",
}

SENTIMENT_CLASS = {
    "positive": "positive",
    "negative": "negative",
    "neutral": "neutral",
    "not_mentioned": "muted",
}


# -----------------------------
# Same light preprocessing used in training
# -----------------------------
class ArabicPreprocessor:
    def __init__(
        self,
        remove_diacritics: bool = True,
        replace_urls: bool = True,
        replace_emails: bool = True,
        replace_mentions: bool = True,
        replace_hashtags: bool = False,
        keep_emojis: bool = True,
        reduce_elongation: bool = True,
    ):
        self.remove_diacritics = remove_diacritics
        self.replace_urls = replace_urls
        self.replace_emails = replace_emails
        self.replace_mentions = replace_mentions
        self.replace_hashtags = replace_hashtags
        self.keep_emojis = keep_emojis
        self.reduce_elongation = reduce_elongation

        self.char_map = str.maketrans({
            "إ": "ا",
            "أ": "ا",
            "آ": "ا",
            "ٱ": "ا",
            "ى": "ي",
            "ک": "ك",
            "ی": "ي",
            "ھ": "ه",
            "ـ": "",
        })
        self.arabic_diacritics = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
        self.control_chars = re.compile(r"[\u0000-\u001F\u007F-\u009F]")
        self.url_pattern = re.compile(r"(http\S+|www\.\S+)")
        self.email_pattern = re.compile(r"\S+@\S+")
        self.mention_pattern = re.compile(r"@\w+")
        self.hashtag_pattern = re.compile(r"#(\w+)")
        self.whitespace_pattern = re.compile(r"\s+")
        self.elongation_pattern = re.compile(r"(.)\1{2,}")
        self.repeated_punct_pattern = re.compile(r"([!?.،؛])\1{2,}")

    def clean(self, text: str) -> str:
        if not isinstance(text, str):
            return ""

        text = unicodedata.normalize("NFKC", text)
        text = self.control_chars.sub(" ", text)
        text = self.url_pattern.sub(" <URL> ", text) if self.replace_urls else text
        text = self.email_pattern.sub(" <EMAIL> ", text) if self.replace_emails else text
        text = self.mention_pattern.sub(" <USER> ", text) if self.replace_mentions else text
        text = self.hashtag_pattern.sub(r" \1 ", text) if self.replace_hashtags else text
        text = text.translate(self.char_map)
        if self.remove_diacritics:
            text = self.arabic_diacritics.sub("", text)
        if self.reduce_elongation:
            text = self.elongation_pattern.sub(r"\1\1", text)
        text = self.repeated_punct_pattern.sub(r"\1\1", text)
        if not self.keep_emojis and emoji is not None:
            text = emoji.replace_emoji(text, replace=" ")
        return self.whitespace_pattern.sub(" ", text).strip()

    def full_preprocess(self, text: str) -> str:
        return self.clean(text)


preprocessor = ArabicPreprocessor()


# -----------------------------
# Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()


def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=-1, keepdims=True)


@torch.no_grad()
def predict_all_aspects(review_text: str) -> Dict[str, Dict[str, Any]]:
    clean_review = preprocessor.full_preprocess(review_text)
    results = {}

    for aspect in ASPECTS:
        prompt = ASPECT_PROMPTS[aspect]
        enc = tokenizer(
            clean_review,
            prompt,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        outputs = model(**enc)
        logits = outputs.logits.detach().cpu().numpy()[0]
        probs = softmax_np(logits)

        pred_id = int(np.argmax(probs))
        pred_label = ID2LABEL[pred_id]
        prob_dict = {ID2LABEL[i]: float(probs[i]) for i in range(len(probs))}

        results[aspect] = {
            "aspect": aspect,
            "pred_label": pred_label,
            "mention_confidence": 1.0 - prob_dict.get("not_mentioned", 0.0),
            "sentiment_confidence": prob_dict.get(pred_label, 0.0),
            "probs": prob_dict,
        }
    return results


def build_output(review_text: str, threshold: float) -> Dict[str, Any]:
    per_aspect = predict_all_aspects(review_text)

    detected_aspects: List[str] = []
    aspect_sentiments: Dict[str, str] = {}
    all_rows = []
    detected_rows = []

    for aspect in ASPECTS:
        info = per_aspect[aspect]
        pred_label = info["pred_label"]
        mention_conf = float(info["mention_confidence"])
        detected = pred_label != "not_mentioned" and mention_conf >= threshold

        row = {
            "Aspect": ASPECT_DISPLAY.get(aspect, aspect),
            "Detected?": "Yes" if detected else "No",
            "Predicted sentiment": SENTIMENT_DISPLAY.get(pred_label, pred_label),
            "Mention confidence": round(mention_conf, 3),
            "Sentiment confidence": round(float(info["sentiment_confidence"]), 3),
        }
        all_rows.append(row)

        if detected:
            detected_aspects.append(aspect)
            aspect_sentiments[aspect] = pred_label
            detected_rows.append(row)

    if not detected_aspects:
        detected_aspects = [OUTPUT_NONE_LABEL]
        aspect_sentiments = {}

    submission_json = {
        "aspects": detected_aspects,
        "aspect_sentiments": aspect_sentiments,
    }

    return {
        "clean_text": preprocessor.full_preprocess(review_text),
        "detected_rows": detected_rows,
        "all_rows": all_rows,
        "submission_json": submission_json,
    }


def render_badges(submission_json: Dict[str, Any]) -> str:
    aspects = submission_json["aspects"]
    sentiments = submission_json["aspect_sentiments"]

    if aspects == [OUTPUT_NONE_LABEL]:
        return "<div class='empty'>No specific aspect was detected. The review may be too short or too general.</div>"

    badges = []
    for aspect in aspects:
        sentiment = sentiments.get(aspect, "neutral")
        css_class = SENTIMENT_CLASS.get(sentiment, "neutral")
        aspect_name = html.escape(ASPECT_DISPLAY.get(aspect, aspect))
        sentiment_name = html.escape(SENTIMENT_DISPLAY.get(sentiment, sentiment))
        badges.append(
            f"<span class='badge {css_class}'><b>{aspect_name}</b> · {sentiment_name}</span>"
        )
    return "<div class='badge-wrap'>" + "".join(badges) + "</div>"


def predict_for_demo(review_text: str, threshold: float):
    if not review_text or not review_text.strip():
        empty_json = {"aspects": [OUTPUT_NONE_LABEL], "aspect_sentiments": {}}
        return (
            "Please enter a review first.",
            [],
            empty_json,
            [],
            "",
        )

    output = build_output(review_text, threshold)
    detected_summary = render_badges(output["submission_json"])
    clean_text = html.escape(output["clean_text"])
    explanation = (
        f"{detected_summary}"
        f"<div class='clean-box'><b>Preprocessed text:</b><br>{clean_text}</div>"
    )

    return (
        explanation,
        output["detected_rows"],
        output["submission_json"],
        output["all_rows"],
        f"Device: {device} · Threshold: {threshold:.2f} · Max tokens: {MAX_LEN}",
    )


DEMO_EXAMPLES = [
    [
        "الخدمة كانت محترمة جدا والمكان رايق، بس الأكل وصل بارد واللحمة ناشفة والسعر مبالغ فيه 😕",
        DEFAULT_THRESHOLD,
    ],
    [
        "el pizza helw awy 😋 bas el service bateeee2.",
        DEFAULT_THRESHOLD,
    ],
    [
        "The app كان جميل و سهل جدا وال payment خلص بسرعة، لكن الدليفري اتأخر والأوردر وصل ناقص.",
        DEFAULT_THRESHOLD,
    ],
    [
        "المكان تحفة والنضافة ممتازة والستاف لطيف، لكن المنيو مفيهوش اختيارات كتير للأطفال.",
        DEFAULT_THRESHOLD,
    ],
    [
        "Honestly wow: clean branch, friendly team, delicious food, بس السعر عالي شوية مقارنة بالحجم.",
        DEFAULT_THRESHOLD,
    ],
    [
        "L'ambiance كانت magnifique والمكان نظيف، لكن le service كان بطيء.",
        DEFAULT_THRESHOLD,
    ],
    [
        "الخدمة كانت excellent والستاف friendly جدا، بس the food was cold والسعر overpriced بالنسبة للجودة.",
        DEFAULT_THRESHOLD,
    ],
    [
        "el burger طعمه حلو جدا، bas السعر expensive والحجم صغير.",
        DEFAULT_THRESHOLD,
    ],
]

CSS = """
.gradio-container {max-width: 1150px !important; margin: auto;}
.hero {
    padding: 22px;
    border-radius: 18px;
    background: linear-gradient(135deg, #f8fafc, #eef2ff);
    border: 1px solid #e5e7eb;
    margin-bottom: 14px;
}
.hero h1 {margin-bottom: 6px;}
.hero p {font-size: 16px; color: #374151;}
.badge-wrap {display: flex; flex-wrap: wrap; gap: 10px; margin: 10px 0 14px 0;}
.badge {padding: 9px 12px; border-radius: 999px; font-size: 15px; border: 1px solid #e5e7eb;}
.positive {background: #ecfdf5; color: #065f46;}
.negative {background: #fef2f2; color: #991b1b;}
.neutral {background: #fffbeb; color: #92400e;}
.muted {background: #f3f4f6; color: #4b5563;}
.empty {padding: 12px; border-radius: 12px; background: #f3f4f6; color: #4b5563;}
.clean-box {padding: 12px; border-radius: 12px; background: #ffffff; border: 1px solid #e5e7eb; margin-top: 10px;}
.footer-note {font-size: 13px; color: #6b7280;}
"""

with gr.Blocks(css=CSS, title="Arabic ABSA Review Intelligence") as demo:
    gr.HTML(
        """
        <div class='hero'>
            <h1>Review Intelligence: Aspect-Based Sentiment Analysis</h1>
            <p>
                This demo reads a customer review, detects the mentioned aspects
                — food, service, price, cleanliness, delivery, ambiance, app experience, and general opinion —
                then predicts sentiment for each detected aspect.
            </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            review_input = gr.Textbox(
                label="Customer review",
                placeholder="Paste an Arabic, English, French, or mixed Franco-Arabic review here...",
                lines=7,
            )
        with gr.Column(scale=1):
            threshold = gr.Slider(
                minimum=0.10,
                maximum=0.90,
                value=DEFAULT_THRESHOLD,
                step=0.05,
                label="Aspect detection threshold",
                info="Lower = more aspects detected. Higher = stricter detection.",
            )
            run_btn = gr.Button("Analyze Review", variant="primary")

    status = gr.Markdown(value="")
    summary_html = gr.HTML(label="Executive summary")

    with gr.Tab("Detected aspects"):
        detected_table = gr.Dataframe(
            headers=[
                "Aspect",
                "Detected?",
                "Predicted sentiment",
                "Mention confidence",
                "Sentiment confidence",
            ],
            label="Detected aspect-level sentiment",
            interactive=False,
        )

    with gr.Tab("Submission JSON"):
        output_json = gr.JSON(label="Model output schema")

    with gr.Tab("All aspect scores"):
        all_scores_table = gr.Dataframe(
            headers=[
                "Aspect",
                "Detected?",
                "Predicted sentiment",
                "Mention confidence",
                "Sentiment confidence",
            ],
            label="All aspect decisions",
            interactive=False,
        )

    gr.Examples(
        examples=DEMO_EXAMPLES,
        inputs=[review_input, threshold],
        label="Demo examples",
        cache_examples=False,
    )

    gr.HTML(
        """
        <p class='footer-note'>
        Tip for your LinkedIn demo: start with a simple Arabic review, then use a mixed English/Arabic example,
        then finish with a Franco-Arabic example to show robustness to informal customer language.
        </p>
        """
    )

    run_btn.click(
        fn=predict_for_demo,
        inputs=[review_input, threshold],
        outputs=[summary_html, detected_table, output_json, all_scores_table, status],
    )
    review_input.submit(
        fn=predict_for_demo,
        inputs=[review_input, threshold],
        outputs=[summary_html, detected_table, output_json, all_scores_table, status],
    )

if __name__ == "__main__":
    demo.queue().launch(share=True, debug=True)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_57/224446967.py:401: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="Arabic ABSA Review Intelligence") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://93599304fe80398e9a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
